# Infernce using sft models

In [1]:
# output drk123 SEL
ADAPTER_DIR="./models/medgemma-pathology-full-lora-1-2_1-3/"
OUPUT_PATH="outputInf/mg_sft_OUTd12_13SEL"


# Get gold annotations:

In [1]:
import pandas as pd
df = pd.read_json('./data/gold_data.json')
df.columns

# Infrerence
       
       

In [4]:
import importlib


# importlib.reload(infer_medgemma_pathology_lora)

from infer_medgemma_pathology_lora import (
load_model_for_inference,
    generate_extraction,
)

MODEL_PATH=  "google/medgemma-27b-text-it"

### SFT / LoRA model inference

In [5]:
model_sft, tokenizer_sft = load_model_for_inference(
    model_id=MODEL_PATH,#"google/medgemma-27b-text-it",
    adapter_dir=ADAPTER_DIR,
    dtype="bf16",
    device_map="auto",
)



[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/808 [00:00<?, ?it/s]

In [1]:
result_sft = generate_extraction(
    report = df["content"].iloc[0],
    model=model_sft,
    tokenizer=tokenizer_sft,
    max_new_tokens=256,
    temperature=0.0,
        return_raw=True,
    return_prompt=True,
)

result_sft["prediction"]

# BATCH inf 

In [10]:
from tqdm.auto import tqdm
import pandas as pd
import torch
import json
import infer_medgemma_pathology_lora as infer_mod

BATCH_SIZE = 10

LOCAL_SYSTEM_PROMPT = """
Tu es un système d'extraction d'information à partir de comptes rendus anatomopathologiques de cancer du sein.

Retourne uniquement un objet JSON valide.
N'invente aucune information.

Si une information est absente, ambiguë ou non déductible:
- utilise "unknown" pour les variables catégorielles
- utilise null pour les variables numériques.

Variables à extraire:
- echantillon: mammectomie, tumorectomie, curage ganglionnaire, biopsie, exérèse chirurgicale, unknown
- numero_gang_prel: nombre de ganglions prélevés/examinés, integer ou null
- numero_gang_att: nombre de ganglions atteints/envahis, integer ou null
- taille: taille tumorale maximale en mm, integer ou null
- diagnostic: histologie principale, string
- grade: SBR1, SBR2, SBR3, unknown
- emboles: oui, non, unknown
- ki67: pourcentage Ki67, integer ou null
- re: positifs, négatifs, unknown
- rp: positifs, négatifs, unknown
- her2: négatif, douteux, positif, unknown
- marges_saines: oui, non, unknown
- rupture_capsulaire: oui, non, unknown

Règles:
- Convertir les cm en mm pour taille.
- HER2 0 ou 1+ = négatif.
- HER2 2+ = douteux.
- HER2 3+ ou amplification FISH/CISH/SISH = positif.
- RE/RP positifs si expression positive, même faible.
- Marges R0/non envahies/saines = oui.
- Marges atteintes/R1/envahies = non.
""".strip()


def _build_user_message_local(report):
    # Prefer the exact module constant if it exists; otherwise use the local copy above.
    system_prompt = getattr(infer_mod, "SYSTEM_PROMPT", LOCAL_SYSTEM_PROMPT)
    return f'''{system_prompt}

Compte rendu:
"""
{report}
"""

Retourne uniquement le JSON.'''


def _build_prompt_with_chat_template_local(report, tokenizer):
    messages = [{"role": "user", "content": _build_user_message_local(report)}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def _get_prompt_for_report(report, tokenizer):
    """Build the prompt without generation and without truncation.

    Priority:
    1. infer_mod.build_prompt_with_chat_template(report, tokenizer), if exported.
    2. local equivalent copied from the training/inference script.
    """
    prompt_builder = getattr(infer_mod, "build_prompt_with_chat_template", None)
    if callable(prompt_builder):
        return prompt_builder(report, tokenizer)
    return _build_prompt_with_chat_template_local(report, tokenizer)


def _parse_prediction(raw_text):
    parser = getattr(infer_mod, "extract_first_json_object", None)
    if callable(parser):
        return parser(raw_text)

    start = raw_text.find("{")
    if start == -1:
        return None

    depth = 0
    in_string = False
    escape = False

    for idx in range(start, len(raw_text)):
        char = raw_text[idx]
        if escape:
            escape = False
            continue
        if char == "\\":
            escape = True
            continue
        if char == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                candidate = raw_text[start : idx + 1]
                try:
                    parsed = json.loads(candidate)
                    return parsed if isinstance(parsed, dict) else None
                except json.JSONDecodeError:
                    return None
    return None


@torch.inference_mode()
def generate_extraction_batch(
    reports,
    model,
    tokenizer,
    max_new_tokens=256,
    temperature=0.0,
):
    """Batched version of generate_extraction with no prompt truncation."""
    reports = list(reports)
    prompts = [_get_prompt_for_report(report, tokenizer) for report in reports]

    # Decoder-only models need left padding for batched generation.
    old_padding_side = getattr(tokenizer, "padding_side", "right")
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    try:
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=False,  
        )

        device = next(model.parameters()).device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        generation_kwargs = dict(
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        if temperature and temperature > 0:
            generation_kwargs.update(do_sample=True, temperature=temperature)
        else:
            generation_kwargs.update(do_sample=False)

        output_ids = model.generate(**inputs, **generation_kwargs)

        # Because all prompts are padded to the same batch width, generated tokens begin
        # at inputs["input_ids"].shape[1] for every row.
        prompt_width = inputs["input_ids"].shape[1]
        generated_ids = output_ids[:, prompt_width:]
        raw_texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

        return [
            {
                "prediction": _parse_prediction(raw_text),
                "raw_text": raw_text,
            }
            for raw_text in raw_texts
        ]
    finally:
        tokenizer.padding_side = old_padding_side


def compare_single_vs_batch_one(report, model, tokenizer, max_new_tokens=256):
    """Optional sanity check: compare one normal inference with batch size 1."""
    single = generate_extraction(
        report=report,
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens,
        temperature=0.0,
        return_raw=True,
        return_prompt=True,
    )
    batch = generate_extraction_batch(
        [report],
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens,
        temperature=0.0,
    )[0]
    local_prompt = _get_prompt_for_report(report, tokenizer)
    return {
        "same_prompt": single.get("prompt") == local_prompt,
        "same_raw_text": single.get("raw_text") == batch.get("raw_text"),
        "same_prediction": single.get("prediction") == batch.get("prediction"),
        "single": single,
        "batch": batch,
    }


gold_cols = [c for c in df.columns if c != "content"]
result_sft = []
reports = df["content"].tolist()

for start in tqdm(
    range(0, len(reports), BATCH_SIZE),
    desc="Running SFT inference",
    total=(len(reports) + BATCH_SIZE - 1) // BATCH_SIZE,
):
    end = min(start + BATCH_SIZE, len(reports))
    batch_reports = reports[start:end]

    batch_results = generate_extraction_batch(
        reports=batch_reports,
        model=model_sft,
        tokenizer=tokenizer_sft,
        max_new_tokens=256,
        temperature=0.0,
    )

    for offset, (report, result) in enumerate(zip(batch_reports, batch_results)):
        i = start + offset
        gold_variables = df.iloc[i][gold_cols].to_dict()

        result_sft.append({
            "row_id": i,
            "report": report,
            "gold_variables": gold_variables,
            "prediction": result["prediction"],
            "raw_text": result.get("raw_text"),
        })

results_sft_df = pd.DataFrame(result_sft)


Running SFT inference:   0%|          | 0/38 [00:00<?, ?it/s]

In [11]:

results_sft_df.to_csv(OUPUT_PATH+"results.csv", index=False)
results_sft_df.to_json(
    OUPUT_PATH+"results.json",
    orient="records",
    force_ascii=False,
    indent=2
)